In [ ]:
with base as (

    select distinct
        ca.client_id,
        ca.campaign_name,

        case
            when ca.com_cus_sgr_desc like '%200%' then 200
            when ca.com_cus_sgr_desc like '%300%' then 300
            when ca.com_cus_sgr_desc like '%400%' then 400
            when ca.com_cus_sgr_desc like '%500%' then 500
        end as nominal

    from _ ca

    join _ cc
        on ca.client_id = cc.client_id

    where cc.campaigns_cnt >= 2

),

client_nominal as (

    select
        client_id,
        nominal,
        count(distinct campaign_name) as hit_cnt

    from base

    where nominal is not null

    group by
        client_id,
        nominal

),

client_max as (

    select
        client_id,
        max(hit_cnt) as max_hit_cnt

    from client_nominal

    group by client_id

)

select
    case
        when max_hit_cnt = 1 then 'все разные'
        else max_hit_cnt || ' раза в один номинал'
    end as group_name,

    count(*) as client_cnt,

    round(
        count(*) * 100.0 / sum(count(*)) over (),
        2
    ) as client_pct

from client_max

group by max_hit_cnt

order by max_hit_cnt;

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# df_repeat - результат SQL запроса
# ожидаемые поля:
# group_name
# client_cnt
# client_pct


plot_df = df_repeat.copy()

plot_df = plot_df.sort_values('client_cnt', ascending=False)


fig, axes = plt.subplots(1, 2, figsize=(16, 6))


# =========================================================
# 1. Количество клиентов
# =========================================================

axes[0].bar(
    plot_df['group_name'],
    plot_df['client_cnt']
)

axes[0].set_title(
    'Распределение клиентов по повторяемости номиналов'
)

axes[0].set_xlabel('Группа')
axes[0].set_ylabel('Количество клиентов')

axes[0].tick_params(axis='x', rotation=15)

axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f'{x:,.0f}'.replace(',', ' ')
    )
)

axes[0].grid(True, axis='y', alpha=0.3)


# =========================================================
# 2. Доля клиентов
# =========================================================

axes[1].bar(
    plot_df['group_name'],
    plot_df['client_pct']
)

axes[1].set_title(
    'Доля клиентов по повторяемости номиналов'
)

axes[1].set_xlabel('Группа')
axes[1].set_ylabel('Доля клиентов, %')

axes[1].tick_params(axis='x', rotation=15)

axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f'{x:.0f}%'
    )
)

axes[1].grid(True, axis='y', alpha=0.3)


fig.suptitle(
    'Повторяемость попадания клиентов в одинаковые номиналы',
    fontsize=16
)

plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


bonus_plot = bonus_by_client.copy()

bonus_plot = bonus_plot.sort_values(
    'client_pct',
    ascending=False
)

colors = [
    'tomato' if x == 'все разные' else 'steelblue'
    for x in bonus_plot['group_name']
]

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    bonus_plot['group_name'],
    bonus_plot['client_pct'],
    color=colors
)

for bar in bars:
    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.5,
        f'{height:.1f}%',
        ha='center',
        va='bottom',
        fontsize=10
    )

ax.set_title('Доля клиентов по повторяемости номиналов')
ax.set_xlabel('Число попаданий в один и тот же номинал')
ax.set_ylabel('Доля клиентов, %')

ax.grid(True, axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=0)

ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x:.0f}%')
)

ax.set_ylim(
    0,
    bonus_plot['client_pct'].max() * 1.15
)

fig.suptitle(
    'Повторяемость попадания клиентов в одинаковые номиналы',
    fontsize=16
)

plt.tight_layout()
plt.show()